In [ ]:
pip install pypdf

In [ ]:
# Importing Libraries
import openai
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS

In [ ]:
# Reading the file and splitting them
pdf_reader = PyPDFLoader("/content/Power BI Ebook.pdf")
documents = pdf_reader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,)
chunks = text_splitter.split_documents(documents)

In [ ]:
# Create embeddings
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
db = FAISS.from_documents(documents=chunks, embedding=embeddings)

In [ ]:
# Calling the model
from langchain_xai import ChatXAI

llm = ChatXAI(
    model='grok-3-latest',
    temperature=0,
    max_tokens=None,
    api_key='YOUR_OPENAI_API_KEY'
)

In [ ]:
#llm.invoke('Who is Roger Federer?')

In [ ]:
# sets up a conversational retrieval chain using LangChain
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template("""Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question.

Chat History:
{chat_history}
Follow up Input: {question}
Standalon question:""")

qa = ConversationalRetrievalChain.from_llm(llm=llm,
                                           retriever=db.as_retriever(),
                                           condense_question_prompt=CONDENSE_QUESTION_PROMPT,
                                           return_source_documents=True,
                                           verbose=False
                                           )

In [ ]:
chat_history=[]
query = """?What are the components of Power BI?"""
result = qa({"question": query, "chat_history": chat_history})
print(result['answer'])

## **RAG Evaluation**

In [ ]:
sample_queries = [
    {
        "question": "What are the main components of Power BI?",
        "answer": "The main components of Power BI are Power BI Desktop, Power BI Service, and Power BI Mobile.",
        "reference": "Power BI has three main components: Desktop (Windows app for creating reports), Service (cloud platform for sharing and collaboration), and Mobile (app for viewing reports on mobile devices)."
    },
    {
        "question": "What are the key features of Power BI Desktop?",
        "answer": "Key features include data connection and transformation, data modeling, interactive visualizations, report publishing, and collaboration tools.",
        "reference": "Power BI Desktop allows users to get data, analyze it with DAX, visualize using 150+ visuals, publish to cloud or on-premises, and collaborate with team members."
    },
    {
        "question": "How does Power BI integrate with Excel?",
        "answer": "Power BI can analyze data in Excel, import Excel data, connect to Excel workbooks, and pin Excel ranges to dashboards.",
        "reference": "Power BI integrates with Excel by enabling analysis in Excel, data import, connection to workbooks, and uploading Excel files for dashboard pinning."
    },
    {
        "question": "What is the Power Query Editor used for?",
        "answer": "Power Query Editor is used for data transformation and cleansing before importing data into models or visualizations.",
        "reference": "Power Query Editor includes tools like ribbon tabs, data preview, query settings, and footer with data stats for transforming and shaping data."
    },
    {
        "question": "How can you group data in Power BI?",
        "answer": "You can group data using the 'Group By' feature in the Transform tab of the Query Editor by selecting columns and specifying aggregate calculations.",
        "reference": "In Power BI's Query Editor, the Group By dialog allows users to select columns and apply aggregate functions to group data."
    }
]


In [ ]:
import pandas as pd
df = pd.DataFrame(sample_queries)
df.head(5)

In [ ]:
retriever = db.as_retriever()

In [ ]:
def process_query(query):
  chat_history = []
  result = qa({"question": query, "chat_history": chat_history})
  relevant_docs = retriever.invoke(query)
  print(result['answer'])
  return result['answer'], relevant_docs

In [ ]:
process_query("What is the Power Query Editor used for?")

#### **RAGAS Framework**

In [ ]:
pip install ragas

In [ ]:
results = []

for _, row in df.iterrows():
  question = row['question']
  ground_truth = row['answer']

  answer, relevant_docs = process_query(question)

  results.append({
      "user_input": question,
      "reference": ground_truth,
      "response": answer,
      "retrieved_contexts": [relevant_docs[0].page_content]
  })

In [ ]:
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

In [ ]:
evaluation_dataset = EvaluationDataset.from_list(results)

In [ ]:
evaluator_llm = LangchainLLMWrapper(llm)

In [ ]:
ragas_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm = evaluator_llm
)

In [ ]:
#ragas_result (grok-3-mini)

In [ ]:
ragas_result

#### **grok-3-mini results:**

To interpret the RAGAS results from your evaluation of the RAG-based pipeline, let's break down the metrics:

- Context Recall (0.5500): This metric indicates that the model correctly recalls relevant information from the provided context about 55% of the time. While it's above 50%, there?s room for improvement, especially as higher recall values are generally preferred for effective context utilization.

- Faithfulness (0.4407): This score suggests that approximately 44% of the responses are consistent with the source context or information. A faithfulness score below 50% indicates that the model may generate responses that diverge from the original context, which can lead to misinformation.

- Factual Correctness (F1 Score of 0.3980): The F1 score combines precision and recall for factual accuracy, and a score of about 40% indicates that less than half of the generated answers are factually correct. This is relatively low and highlights a significant area for improvement.

#### **grok-3-latest results**

- Context Recall (0.7500): This score indicates that the model now correctly recalls relevant context information 75% of the time. This is a strong improvement and suggests that the model is effectively utilizing the context provided.

- Faithfulness (0.6420): With a faithfulness score of 64.2%, the model is generating responses that are consistent with the source context more than 60% of the time. This is a notable improvement, indicating greater reliability in the responses.

- Factual Correctness (F1 Score of 0.5940): An F1 score of approximately 59.4% suggests that the model is now producing factually correct answers nearly 60% of the time. This is a significant enhancement and indicates that the model is becoming more accurate.

#### **n-Gram Metrics**

- BLEU
- ROUGE

In [ ]:
pip install rouge

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge

In [ ]:
rouge = Rouge()
bleu = []
rouge_one = []

In [ ]:
for item in results:
  reference = [item["reference"].split()]
  hypothesis = item["response"].split()
  bleu.append(sentence_bleu(reference, hypothesis))

  scores = rouge.get_scores(item["response"], item["reference"])[0]
  rouge_one.append(scores["rouge-1"]["f"])

In [ ]:
bleu

In [ ]:
rouge_one

In [ ]:
#df.to_csv('queries.csv')

#### **Model-based metrics**

- BARTScore
- BERTScore

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer
from bert_score import BERTScorer
import torch

In [ ]:
# Initialize the BART model and tokenizer
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')

# Initialize the BERTScorer
bert_scorer = BERTScorer(lang='en',rescale_with_baseline=True)

In [ ]:
bert_scores = []
bart_scores = []

for item in results:
  inputs = bart_tokenizer(item["response"], return_tensors='pt', truncation=True, padding=True )
  with torch.no_grad():
    bart_score = bart_model(**inputs).logits
  bart_scores.append(bart_score.mean().item())

  P, R, F1 = bert_scorer.score([item["response"]], [item["reference"]])
  bert_scores.append(F1.numpy().mean())

In [ ]:
bart_scores

In [ ]:
bert_scores